# Z5008 Big Data Lab — Practical Assignment
**Student Roll Number:** ZDA25M009

---
## Task 1: Dataset Design and Selection
**Domain:** E-commerce Orders

### Schema & Data Dictionary
- **`order_id` (String):** Unique identifier for each order.
- **`user_id` (String):** Identifier for the customer.
- **`order_date` (Timestamp):** Date and time the order was placed.
- **`product_category` (String):** Category of the product (e.g., Electronics, Fashion).
- **`order_amount` (Double):** Total amount of the order in USD.
- **`status` (String):** Order status (Completed, Pending, Cancelled, Refunded).
- **`payment_method` (String):** Method used for payment.

### Data Quality Rules
1. **Rule 1 (Amount):** `order_amount >= 0`
2. **Rule 2 (Date):** `order_date` must be within the year 2025 or 2026.
3. **Rule 3 (Status):** `status` must be one of `['Completed', 'Pending', 'Cancelled', 'Refunded']`.


In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# Ensure data directory exists
os.makedirs("../data/raw", exist_ok=True)

print("Generating synthetic e-commerce data (50,000+ rows)...")
n_rows = 55000
np.random.seed(42)

categories = ["Electronics", "Fashion", "Home", "Sports", "Books", "Toys"]
statuses = ["Completed", "Pending", "Cancelled", "Refunded", "INVALID_STATUS"] # Injected some invalid for DQ checks
payment_methods = ["Credit Card", "PayPal", "Crypto", "Bank Transfer"]

start_date = datetime(2025, 1, 1)
dates = [start_date + timedelta(minutes=random.randint(0, 525600)) for _ in range(n_rows)]
# Inject some future invalid dates
dates[0] = datetime(2030, 1, 1)

amounts = np.round(np.random.uniform(-50, 1000, n_rows), 2) # Injected negative amounts for DQ checks

df_raw = pd.DataFrame({
    "order_id": [f"ORD-{i:06d}" for i in range(n_rows)],
    "user_id": [f"USR-{random.randint(1000, 9999)}" for _ in range(n_rows)],
    "order_date": dates,
    "product_category": [random.choice(categories) for _ in range(n_rows)],
    "order_amount": amounts,
    "status": [random.choice(statuses) for _ in range(n_rows)],
    "payment_method": [random.choice(payment_methods) for _ in range(n_rows)]
})

raw_path = "../data/raw/ecommerce_orders.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Data successfully written to {raw_path}")

## Task 2: Ingestion + Lakehouse Storage
Initializing Spark with Delta Lake support and MinIO connectivity.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("Z5008-Practical-ZDA25M009")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)

spark = configure_spark_with_delta_pip(
    builder, 
    extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4", "com.amazonaws:aws-java-sdk-bundle:1.12.262"]
).getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session Initialized.")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql import functions as F

# 2.1 Ingest raw data into Spark with explicit schema
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("product_category", StringType(), True),
    StructField("order_amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True)
])

df_spark_raw = spark.read.csv(raw_path, header=True, schema=schema)

# 2.2 Write a Bronze table partitioned by date
df_bronze = df_spark_raw.withColumn("order_date_only", F.to_date("order_date"))
BRONZE_PATH = "s3a://warehouse/bronze/ecommerce/"

df_bronze.write.format("delta").mode("overwrite").partitionBy("order_date_only").save(BRONZE_PATH)
print("Bronze table written successfully.")

In [ ]:
# 2.3 Create a Silver table (clean and standardize)
df_silver = spark.read.format("delta").load(BRONZE_PATH)

# Cleaning: Deduplication and handling nulls
df_silver_clean = df_silver.dropDuplicates(["order_id"]) \
                           .fillna({"order_amount": 0.0, "status": "Unknown"}) \
                           .withColumn("order_amount", F.round(F.col("order_amount"), 2))

SILVER_PATH = "s3a://warehouse/silver/ecommerce/"
df_silver_clean.write.format("delta").mode("overwrite").save(SILVER_PATH)
print("Silver table written successfully.")

In [ ]:
# 2.4 Create a Gold table (aggregated KPIs)
df_gold = df_silver_clean.filter(F.col("status") == "Completed") \
    .groupBy("order_date_only", "product_category") \
    .agg(
        F.sum("order_amount").alias("total_revenue"),
        F.count("order_id").alias("total_orders")
    )

GOLD_PATH = "s3a://warehouse/gold/ecommerce/"
df_gold.write.format("delta").mode("overwrite").save(GOLD_PATH)
print("Gold table written successfully.")

## Task 3: Spark Transformations + Validations

In [ ]:
from pyspark.sql.window import Window

df_silver = spark.read.format("delta").load(SILVER_PATH)

# 3.1 Implement 5 meaningful transformations
# T1: Extract date parts (Year, Month)
df_t1 = df_silver.withColumn("order_year", F.year("order_date")) \
                 .withColumn("order_month", F.month("order_date"))

# T2: Bucketing amounts into categories
df_t2 = df_t1.withColumn("amount_bucket", 
    F.when(F.col("order_amount") < 50, "Low")
     .when(F.col("order_amount") < 200, "Medium")
     .otherwise("High")
)

# T3: Lookup Dimension Join (Creating a dummy dimension DataFrame)
dim_data = [("Electronics", "Tech"), ("Fashion", "Apparel"), ("Home", "Living")]
dim_schema = StructType([StructField("product_category", StringType()), StructField("broad_category", StringType())])
df_dim = spark.createDataFrame(dim_data, schema=dim_schema)
df_t3 = df_t2.join(df_dim, on="product_category", how="left")

# T4: Window Function (Running Total of Sales per User)
window_spec = Window.partitionBy("user_id").orderBy("order_date")
df_t4 = df_t3.withColumn("cumulative_spend", F.sum("order_amount").over(window_spec))

# T5: Conditional Flagging (High Value User Flag)
df_transformed = df_t4.withColumn("is_high_value_order", F.col("order_amount") > 500)
df_transformed.show(5)

In [ ]:
# 3.2 Data Quality Checks
print("--- Data Quality Checks ---")

# Rule 1: Non-negative amounts
violating_amounts = df_transformed.filter(F.col("order_amount") < 0)
print(f"Rule 1 Violations (Negative Amounts): {violating_amounts.count()}")
if violating_amounts.count() > 0:
    violating_amounts.select("order_id", "order_amount").show(5)
# Decision: Flag/Drop. We will filter them out for final analytics.

# Rule 2: Valid Date Range (2025-2026)
violating_dates = df_transformed.filter(~F.col("order_year").isin(2025, 2026))
print(f"Rule 2 Violations (Invalid Dates): {violating_dates.count()}")
if violating_dates.count() > 0:
    violating_dates.select("order_id", "order_date").show(5)
# Decision: Drop rows with invalid dates.

# Rule 3: Allowed Categories
allowed_statuses = ["Completed", "Pending", "Cancelled", "Refunded"]
violating_status = df_transformed.filter(~F.col("status").isin(allowed_statuses))
print(f"Rule 3 Violations (Invalid Status): {violating_status.count()}")
if violating_status.count() > 0:
    violating_status.select("order_id", "status").show(5)
# Decision: Flag and clean these rows in production, here we will filter them.

# Apply Filtering based on DQ rules
df_cleaned_final = df_transformed.filter(
    (F.col("order_amount") >= 0) & 
    (F.col("order_year").isin(2025, 2026)) &
    (F.col("status").isin(allowed_statuses))
)

In [ ]:
# 3.3 Analytical Outputs
os.makedirs("../outputs", exist_ok=True)

# Output 1: Top 10 Spending Users
top_users = df_cleaned_final.filter(F.col("status") == "Completed") \
    .groupBy("user_id").agg(F.sum("order_amount").alias("total_spent")) \
    .orderBy(F.desc("total_spent")).limit(10)

top_users.write.mode("overwrite").parquet("s3a://warehouse/gold/outputs/top_10_users.parquet")
top_users.toPandas().to_csv("../outputs/top_10_users.csv", index=False)

# Output 2: Daily Revenue Trend
daily_trend = df_cleaned_final.filter(F.col("status") == "Completed") \
    .groupBy("order_date_only").agg(F.sum("order_amount").alias("daily_revenue")) \
    .orderBy("order_date_only")

daily_trend.write.mode("overwrite").parquet("s3a://warehouse/gold/outputs/daily_revenue_trend.parquet")
daily_trend.toPandas().to_csv("../outputs/daily_revenue_trend.csv", index=False)

print("Analytical outputs saved successfully.")

## Task 4: Spark SQL + Performance/Tuning

In [ ]:
# 4.1 Register tables and query with Spark SQL
spark.read.format("delta").load(BRONZE_PATH).createOrReplaceTempView("bronze_ecommerce")
spark.read.format("delta").load(SILVER_PATH).createOrReplaceTempView("silver_ecommerce")
spark.read.format("delta").load(GOLD_PATH).createOrReplaceTempView("gold_ecommerce")

print("--- SQL Query 1: Categories with Total Revenue > $1M ---")
spark.sql("""
    SELECT product_category, SUM(total_revenue) as total_category_revenue
    FROM gold_ecommerce
    GROUP BY product_category
    HAVING total_category_revenue > 1000000
""").show()

print("--- SQL Query 2: Window Function (Rank Users by Total Spend per Category) ---")
spark.sql("""
    WITH UserCategorySpend AS (
        SELECT user_id, product_category, SUM(order_amount) as total_spent
        FROM silver_ecommerce
        WHERE status = 'Completed'
        GROUP BY user_id, product_category
    )
    SELECT *, RANK() OVER (PARTITION BY product_category ORDER BY total_spent DESC) as rank
    FROM UserCategorySpend
    WHERE total_spent IS NOT NULL
    LIMIT 10
""").show()

In [ ]:
# 4.2 Performance Evidence
# Explain Plan before caching vs after caching
print("Explain Plan (Without Cache):")
df_query = spark.sql("SELECT product_category, SUM(order_amount) FROM silver_ecommerce GROUP BY product_category")
df_query.explain()

print("\nExplain Plan (With Cache):")
spark.catalog.cacheTable("silver_ecommerce")
df_query_cached = spark.sql("SELECT product_category, SUM(order_amount) FROM silver_ecommerce GROUP BY product_category")
df_query_cached.explain()

### 4.3 Short Reflection
The biggest bottleneck observed was the repeated reading and shuffling of data from MinIO when executing multiple aggregation queries on the `silver_ecommerce` table. To mitigate this, I applied data caching (`spark.catalog.cacheTable`), which loads the dataset into distributed memory across the worker nodes. As evidenced by the `explain()` plan, caching replaces the physical MinIO file scan with an `InMemoryTableScan`, significantly reducing I/O latency and speeding up downstream Spark SQL queries.
